In [2]:
import os
from torchmdnet.datasets import PCQM4MV2_Dihedral2

In [3]:
pcq_data = PCQM4MV2_Dihedral2(root='/data/protein/SKData/DenoisingData/pcq', sdf_path=None, dihedral_angle_noise_scale=2, position_noise_scale=0.04, composition=True, decay=False, addh=True)

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem

def rdkit_generate_and_optimize(test_mol):
    try:
        test_mol.RemoveConformer(0)
        cids = AllChem.EmbedMultipleConfs(test_mol, numConfs=1, numThreads=1, pruneRmsThresh=0.1, maxAttempts=5, useRandomCoords=False)
        if len(cids) < 1:
            # rdkit_failed_cnt += 1
            print('rdkit generate fail')
        else:
            AllChem.MMFFOptimizeMoleculeConfs(test_mol, numThreads=1)
    except Exception as e:
        print(f'exeption captured {e}')

In [5]:
import lmdb
import pickle

root = '/data/protein/SKData/DenoisingData/pcq'
MOL_LST = lmdb.open(f'{root}/MOL_LMDB', readonly=True, subdir=True, lock=False)

In [15]:
import time
test_number = 100
time1 = time.time()
for i in range(test_number):
    with MOL_LST.begin(write=False) as txn:
        mol = pickle.loads(txn.get(str(i).encode()))
        rdkit_generate_and_optimize(mol)
time2 = time.time()
print(f'rdkit time: {time2-time1}')

rdkit generate fail
rdkit time: 2.773611307144165


In [16]:
import torch
time1 = time.time()

for i in range(test_number):
    idx = torch.tensor([i])
    mol = pcq_data[idx]
time2 = time.time()
print(f'torchmd time: {time2-time1}')

torchmd time: 0.09314465522766113
